In [4]:
import tensorflow as tf
from tensorflow import keras
import urllib.request
import zipfile
import os
from keras.applications.inception_v3 import InceptionV3
from keras.models import Model
from keras.callbacks import ModelCheckpoint
from keras.layers import Flatten, Dense, Dropout
from keras.optimizers import *
from keras.preprocessing.image import ImageDataGenerator
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import random

In [12]:

training_dir = 'database_images/test'
test_dir = 'database_images/train'

train_datagen = ImageDataGenerator(
    rescale = 1/255,
    #rotation_range=45,
    #width_shift_range=0.2,
    #height_shift_range=0.2,
    # shear_range=0.2,
    #zoom_range=0.2,
    #horizontal_flip=True,
    # fill_mode='nearest'
    )

train_generator = train_datagen.flow_from_directory(
    training_dir,
    target_size=(300, 300),
    class_mode='categorical',
    batch_size=16
)

test_datagen = ImageDataGenerator(rescale=1/255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(300, 300),
    class_mode = 'categorical',
    batch_size=16
)

weights_url = "https://storage.googleapis.com/mledu-datasets/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5"

weights_file = "inception_v3.h5"
urllib.request.urlretrieve(weights_url, weights_file)

pre_trained_model = InceptionV3(input_shape=(300, 300, 3),
                                include_top=False,
                                weights=None)

pre_trained_model.load_weights(weights_file)

#pre_trained_model.summary()

for layer in pre_trained_model.layers:
  layer.trainable = False

last_layer = pre_trained_model.get_layer('mixed7')
print('last layer output shape: ', last_layer.output_shape)
last_output = last_layer.output

x = Flatten()(last_output)
x = Dense(256, activation='relu')(x)
# x = Dense(128, activation='relu')(x)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.2)(x)
#x = Dense(8, activation='softmax')(x)
x = Dense(6, activation='softmax')(x)

model = Model(pre_trained_model.input, x)

# model.summary()

model.compile(optimizer=Nadam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['acc'])

checkpoint = ModelCheckpoint("model.h5", save_best_only=True)

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=[checkpoint]
)

Found 100 images belonging to 6 classes.
Found 388 images belonging to 6 classes.
last layer output shape:  (None, 17, 17, 768)
Epoch 1/10


2024-10-01 22:11:28.826849: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


7/7 [==============================] - ETA: 0s - loss: 1.8680 - acc: 0.1300

2024-10-01 22:11:39.649299: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


7/7 [==============================] - 29s 4s/step - loss: 1.8680 - acc: 0.1300 - val_loss: 1.6959 - val_acc: 0.3840
Epoch 2/10
7/7 [==============================] - 25s 4s/step - loss: 1.7073 - acc: 0.3500 - val_loss: 1.6905 - val_acc: 0.3479
Epoch 3/10
7/7 [==============================] - 25s 4s/step - loss: 1.7503 - acc: 0.3200 - val_loss: 1.6268 - val_acc: 0.4253
Epoch 4/10
7/7 [==============================] - 25s 4s/step - loss: 1.5772 - acc: 0.3800 - val_loss: 1.6177 - val_acc: 0.3376
Epoch 5/10
7/7 [==============================] - 25s 4s/step - loss: 1.5537 - acc: 0.3800 - val_loss: 1.5350 - val_acc: 0.4253
Epoch 6/10
7/7 [==============================] - 25s 4s/step - loss: 1.4589 - acc: 0.4500 - val_loss: 1.4478 - val_acc: 0.4820
Epoch 7/10
7/7 [==============================] - 24s 4s/step - loss: 1.4560 - acc: 0.4000 - val_loss: 1.5589 - val_acc: 0.4175
Epoch 8/10
7/7 [==============================] - 25s 4s/step - loss: 1.3973 - acc: 0.5700 - val_loss: 1.4229 - val

# Siamese network

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from keras.preprocessing.image import ImageDataGenerator
import random
# Função para criar a arquitetura da Rede Siamesa
def create_siamese_model(input_shape):
    input = layers.Input(input_shape)
    
    # Arquitetura da sub-rede (CNN simples)
    x = layers.Conv2D(64, (5,5), activation='tanh')(input)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, (3,3), activation='sigmoid')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, (3,3), activation='sigmoid')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, (3,3), activation='sigmoid')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(16, activation='sigmoid')(x)
    
    model = models.Model(input, x)
    return model

# Definir a função de distância entre as duas saídas
def euclidean_distance(vectors):
    (featA, featB) = vectors
    sum_squared = tf.reduce_sum(tf.square(featA - featB), axis=1, keepdims=True)
    return tf.sqrt(tf.maximum(sum_squared, tf.keras.backend.epsilon()))

# Input shape (28x28, como no dataset Omniglot)
input_shape = (300, 300, 3)

# Criação do modelo siamesa
base_network = create_siamese_model(input_shape)

# Definir as duas entradas
input_a = layers.Input(shape=input_shape)
input_b = layers.Input(shape=input_shape)

# Extração das características para as duas entradas
feat_a = base_network(input_a)
feat_b = base_network(input_b)

# Distância entre as saídas das redes
distance = layers.Lambda(euclidean_distance)([feat_a, feat_b])

# Modelo Siamesa completo
model = models.Model(inputs=[input_a, input_b], outputs=distance)

# Compilar o modelo
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

2024-10-02 09:39:29.430267: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-02 09:39:29.629143: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-02 09:39:30.803535: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-10-02 09:39:34.263938: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [2]:
def generate_image_pairs(generator, num_pairs=1000):
    """
    Gera pares de imagens e os rótulos correspondentes, onde:
    - generator: Gerador de imagens (como o gerado pelo ImageDataGenerator)
    - num_pairs: Número de pares que você quer gerar
    """
    pairs = []
    labels = []
    
    X, y = generator.next()  # Carrega um batch de imagens e rótulos
    n_classes = len(np.unique(y))  # Número de classes no dataset
    class_indices = [np.where(y == i)[0] for i in range(n_classes)]  # Índices por classe
    
    for _ in range(num_pairs):
        if random.choice([True, False]):
            # Mesma classe
            class_id = random.randint(0, n_classes - 1)
            idx1, idx2 = random.sample(list(class_indices[class_id]), 2)
            pairs.append([X[idx1], X[idx2]])
            labels.append(1)  # Mesmo rótulo
        else:
            # Classes diferentes
            class_id1, class_id2 = random.sample(range(n_classes), 2)
            idx1 = random.choice(class_indices[class_id1])
            idx2 = random.choice(class_indices[class_id2])
            pairs.append([X[idx1], X[idx2]])
            labels.append(0)  # Classes diferentes
    
    return np.array(pairs), np.array(labels)

# Gerando 1000 pares de imagens a partir do diretório de treinamento
# X_pairs, y_pairs = generate_image_pairs(train_generator, num_pairs=1000)

In [3]:
training_dir = 'database_images/test'
test_dir = 'database_images/train'

train_datagen = ImageDataGenerator(
    rescale = 1/255,
    #rotation_range=45,
    #width_shift_range=0.2,
    #height_shift_range=0.2,
    # shear_range=0.2,
    #zoom_range=0.2,
    #horizontal_flip=True,
    # fill_mode='nearest'
    )

train_generator = train_datagen.flow_from_directory(
    training_dir,
    target_size=(300, 300),
    class_mode='categorical',
    # class_mode='sparse',
    shuffle=True,
    batch_size=16
)

test_datagen = ImageDataGenerator(rescale=1/255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(300, 300),
    class_mode = 'categorical',
    # class_mode = 'sparse',
    shuffle=True,
    batch_size=16
)

X_pairs, y_pairs = generate_image_pairs(train_generator, num_pairs=1000)

# Ajustando o modelo com os dados gerados
# X_pairs[:, 0] e X_pairs[:, 1] são os pares de imagens
model.fit([X_pairs[:, 0], X_pairs[:, 1]], y_pairs, batch_size=16, epochs=10)

Found 100 images belonging to 6 classes.
Found 388 images belonging to 6 classes.
Epoch 1/10
63/63 [==============================] - 130s 2s/step - loss: 4.3200 - accuracy: 0.4630
Epoch 2/10
63/63 [==============================] - 131s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 3/10
63/63 [==============================] - 132s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 4/10
63/63 [==============================] - 132s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 5/10
63/63 [==============================] - 131s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 6/10
63/63 [==============================] - 131s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 7/10
63/63 [==============================] - 133s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 8/10
63/63 [==============================] - 134s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 9/10
62/63 [============================>.] - ETA: 2s - loss: 4.3463 - accuracy: 0.4607

KeyboardInterrupt: 